# 01 · Análisis exploratorio (datos limpios de silver/gold)
Cada gráfico responde una pregunta operativa del CCO de LATAM. Solo se usan datos reales (ANAC VRA + METAR IEM);
nunca las réplicas de `scale-test/`.

In [ ]:
import os, sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
os.environ.setdefault("LATAM_MODO", "local")          # "aws" en el clúster (lee s3a://$LAKE_BUCKET)
os.environ.setdefault("LATAM_LAKE_LOCAL", str(RAIZ / "data" / "lake_muestra"))
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from latam_ops.comun.config import lake
from latam_ops.comun.sesion_spark import crear_sesion
spark = crear_sesion("notebook")
lk = lake()

In [ ]:
vuelos = spark.read.parquet(lk.ruta("silver_vuelos")).where("es_latam")
feats = spark.read.parquet(lk.ruta("gold_features"))
print("vuelos LATAM:", vuelos.count(), "| filas de modelado:", feats.count())
vuelos.groupBy("empresa_icao").count().orderBy(F.desc("count")).show()

## P1. ¿Qué tan frecuente es salir con ≥ 15 y ≥ 60 min de demora? (línea base)

In [ ]:
lb = vuelos.where("NOT cancelado AND y_demora15 IS NOT NULL").agg(
    F.avg("y_demora15").alias("tasa15"), F.avg("y_demora60").alias("tasa60"), F.avg(F.col("cancelado").cast("int")).alias("x"),
    F.expr("percentile_approx(demora_salida_min, 0.5)").alias("mediana")).first()
print(f"Salidas con >=15 min: {lb.tasa15:.1%} | >=60 min: {lb.tasa60:.1%} | mediana de demora: {lb.mediana} min")
pdf = vuelos.where("demora_salida_min BETWEEN -30 AND 180").select("demora_salida_min").toPandas()
ax = pdf["demora_salida_min"].plot.hist(bins=70, log=True, figsize=(8, 4))
ax.axvline(15, color="orange", label="ámbar 15 min"); ax.axvline(60, color="red", label="rojo 60 min")
ax.set_xlabel("Demora de salida (min)"); ax.set_ylabel("Vuelos (escala log)"); ax.legend(); plt.show()

## P2. ¿A qué hora del día se concentran las demoras?

In [ ]:
p2 = feats.groupBy("hora_local").agg(F.avg("y_demora15").alias("tasa15"), F.count("*").alias("n")).orderBy("hora_local").toPandas()
ax = p2.plot.bar(x="hora_local", y="tasa15", figsize=(9, 4), legend=False)
ax.set_ylabel("Tasa de demora >= 15 min"); ax.set_xlabel("Hora local programada"); plt.show()
print("Hora con mayor tasa:", p2.loc[p2.tasa15.idxmax()].to_dict())

## P3. ¿Empeora la puntualidad con malas condiciones meteorológicas a T−2 h (VFR → LIFR)?

In [ ]:
p3 = feats.groupBy("o_categoria_vuelo").agg(F.avg("y_demora15").alias("tasa15"), F.avg("y_demora60").alias("tasa60"),
                                             F.count("*").alias("n")).toPandas().set_index("o_categoria_vuelo").reindex(["VFR", "MVFR", "IFR", "LIFR"])
print(p3)
p3[["tasa15", "tasa60"]].plot.bar(figsize=(7, 4)); plt.ylabel("Tasa"); plt.title("Demora según categoría de vuelo en el origen a T-2 h"); plt.show()

## P4. ¿La red ya atrasada en el origen (2 h previas a T−2 h) anticipa la demora? (propagación)

In [ ]:
p4 = (feats.withColumn("red", F.when(F.col("red_frac_atrasadas").isNull(), "sin salidas")
          .when(F.col("red_frac_atrasadas") < 0.1, "<10%").when(F.col("red_frac_atrasadas") < 0.3, "10-30%").otherwise(">=30%"))
      .groupBy("red").agg(F.avg("y_demora15").alias("tasa15"), F.count("*").alias("n")).toPandas().set_index("red"))
print(p4)
p4["tasa15"].plot.bar(figsize=(6, 4)); plt.ylabel("Tasa de demora >= 15"); plt.xlabel("% de salidas atrasadas en el origen"); plt.show()

## P5. ¿Qué aeródromos de origen concentran el problema?

In [ ]:
p5 = (feats.groupBy("origen_icao").agg(F.count("*").alias("n"), F.avg("y_demora15").alias("tasa15"))
      .where("n >= 200").orderBy(F.desc("n")).limit(15).toPandas())
ax = p5.plot.barh(x="origen_icao", y="tasa15", figsize=(7, 5), legend=False); ax.set_xlabel("Tasa de demora >= 15"); plt.show()

**Conclusiones del EDA:** escribir aquí los hallazgos accionables con las cifras impresas arriba (hora, clima, red, aeródromo).